#This file does an initial analysis of the Responsa metadata plus word length data. The initial data is in /content/gdrive/MyDrive/Responsa/full_stats_with_missing.xlsx". This file is merged with /content/gdrive/MyDrive/Responsa/geonim_len_stats.xlsx"), which contains the data for responsa authored by geonim. The code then fills in birth and location data where it is missing, estimates the year in which a responsa is written from its order within a volume, filters the data to eliminate non-responsa (such as notes), assigns a region to each observation from its country. The file then builds a table of summary statistics and saves the data for further analysis as /content/gdrive/MyDrive/Responsa/bi_plus_geonim.csv.

In [1]:
import pandas as pd
import numpy as np
import sklearn as sk
!pip install python-docx


from google.colab import drive
drive.mount('/content/gdrive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.2 MB/s eta 0:00:00
Mounted at /content/gdrive


In [2]:
df = pd.read_excel("/content/gdrive/MyDrive/Responsa/full_stats_with_missing.xlsx")
#reading in Geonim data
df_new = pd.read_excel("/content/gdrive/MyDrive/Responsa/geonim_len_stats.xlsx")  # Add `sheet_name='Sheet1'` if needed

df['source'] = 'dataset_1'
df_new['source'] = 'dataset_2'


#appending Geonim data to main data
df = pd.concat([df, df_new], ignore_index=True)


In [3]:

# Add an index column showing the order of each responsum within (author_name, book_name)
df['unit_index'] = df.groupby(['author_name', 'book_name']).cumcount() + 1

In [4]:
# Get unique author_name values where birth_year is missing
missing_birth_year_authors = df.loc[df['birth_year'].isna(), 'author_name'].unique()

print(missing_birth_year_authors)


["ר' יעקב ישראל חגיז" "ר' יוסף חיים אל חכם" "ר' אריה ליבוש ליפשיץ"
 "ר' משה יהושע יהודה לייב דיסקין" "ר' יוסף משאש" "ר' יהודה אריה גרוסנס"
 "ר' פסח אליהו פלק" "ר' משה גרינוולד" "ר' אריה צבי פרומר"
 "ר' יואב יהושע וינגרטן" "ר' שמואל יצחק שור" "ר' ברוך לאווסקי"
 "ר' מאיר די בוטון" "ר' מרדכי זיסקינד רוטנברג" "ר' שמואל ענגיל"
 "ר' יוסף אליהו הנקין" "ר' שלום משאש" "ר' יהודה הרצל הנקין"
 "ר' יצחק אבולעפיה" "ר' שמואל אבוהב" "ר' שלמה יהודה טאבאק"
 "ר' שלמה זלמן ליפשיץ" "ר' יעקב אבן יחיא" "ר' יחיאל באסן" "ר' סיני ספיר"
 "ר' יחזקאל קצנלנפוגן" "ר' יוסף אירגאס" "ר' אברהם פיוטרקובסקי"
 "ר' רפאל כץ" "ר' יעקב מאיר פאדווא" "ר' אברהם ליטש רוזנבוים"
 "ר' ישראל בעלסקי" "ר' יעקב ברוכין" "ר' מרדכי פארהאנד" "ר' יחיאל העליר"
 "ר' אברהם ישראל זאבי" "ר' אליהו בקשי דורון" "ר' ישראל ניסן קופרשטוך"
 "ר' בן ציון בלום" "ר' אהרן לפפא" 'ר\' חיים ב"ר שמחה הכהן רפפורט'
 "ר' משה שטרן" 'ר\' ראובן ב"ר בצלאל הכהן רפפורט'
 "ר' אריה יהודה ליב תאומים" 'ר\' שלמה ב"ר יצחק לבית לוי' "ר' מאיר אריק"]


In [5]:
# Filling in missing birth years and countries Dictionary of author_name in Hebrew mapped to birth_year and country
data = {
    "ר' יעקב ישראל חגיז": (1620, 'ארץ ישראל'),
    "ר' יוסף חיים אל חכם": (1851, 'עיראק'),
    "ר' אריה ליבוש ליפשיץ": (1760, 'פולין'),
    "ר' משה יהושע יהודה לייב דיסקין": (1817, 'ליטא, ארץ ישראל'),
    "ר' יוסף משאש": (1892, 'מרוקו, ישראל'),
    "ר' יהודה אריה גרוסנס": (1870, 'פולין'),
    "ר' פסח אליהו פלק": (1878, 'פולין'),
    "ר' משה גרינוולד": (1853, 'הונגריה'),
    "ר' אריה צבי פרומר": (1884, 'פולין'),
    "ר' יואב יהושע וינגרטן": (1845, 'פולין'),
    "ר' שמואל יצחק שור": (1873, 'פולין'),
    "ר' ברוך לאווסקי": (1867, 'פולין'),
    "ר' מאיר די בוטון": (1575, 'טורקיה'),
    "ר' מרדכי זיסקינד רוטנברג": (1830, 'פולין'),
    "ר' שמואל ענגיל": (1853, 'פולין'),
    "ר' יוסף אליהו הנקין": (1881, 'בלארוס, ארה"ב'),
    "ר' שלום משאש": (1909, 'מרוקו, ישראל'),
    "ר' יהודה הרצל הנקין": (1945, 'ארה"ב, ישראל'),
    "ר' יצחק אבולעפיה": (1824, 'ארץ ישראל'),
    "ר' שמואל אבוהב": (1610, 'איטליה'),
    "ר' שלמה יהודה טאבאק": (1832, 'הונגריה'),
    "ר' שלמה זלמן ליפשיץ": (1765, 'פולין'),
    "ר' יעקב אבן יחיא": (1470, 'פורטוגל'),
    "ר' יחיאל באסן": (1653, 'איטליה'),
    "ר' סיני ספיר": (1820, 'ליטא'),
    "ר' יחזקאל קצנלנפוגן": (1668, 'פולין'),
    "ר' יוסף אירגאס": (1685, 'איטליה'),
    "ר' אברהם פיוטרקובסקי": (1897, 'פולין'),
    "ר' רפאל כץ": (1770, 'פולין'),
    "ר' יעקב מאיר פאדווא": (1890, 'הונגריה, אנגליה'),
    "ר' אברהם ליטש רוזנבוים": (1860, 'הונגריה'),
    "ר' ישראל בעלסקי": (1938, 'ארה"ב'),
    "ר' יעקב ברוכין": (1879, 'פולין'),
    "ר' מרדכי פארהאנד": (1872, 'הונגריה'),
    "ר' יחיאל העליר": (1814, 'הונגריה'),
    "ר' אברהם ישראל זאבי": (1869, 'פולין'),
    "ר' אליהו בקשי דורון": (1941, 'ישראל'),
    "ר' ישראל ניסן קופרשטוך": (1900, 'פולין'),
    "ר' בן ציון בלום": (1908, 'הונגריה'),
    "ר' אהרן לפפא": (1790, 'פולין'),
    "ר' חיים ב\"ר שמחה הכהן רפפורט": (1700, 'פולין'),
    "ר' משה שטרן": (1914, 'הונגריה, ארה"ב'),
    "ר' ראובן ב\"ר בצלאל הכהן רפפורט": (1750, 'פולין'),
    "ר' אריה יהודה ליב תאומים": (1813, 'הונגריה'),
    "ר' שלמה ב\"ר יצחק לבית לוי": (1020, 'צרפת'),
    "ר' מאיר אריק": (1855, 'פולין')
}


# Hebrew to English mapping for countries/regions
country_translation = {
    'ארץ ישראל': 'Israel',
    'עיראק': 'Iraq',
    'פולין': 'Poland',
    'ליטא': 'Lithuania',
    'ליטא, ארץ ישראל': 'Lithuania',
    'מרוקו, ישראל': 'Morocco',
    'מרוקו': 'Morocco',
    'הונגריה': 'Hungary',
    'איטליה': 'Italy',
    'בלארוס, ארה"ב': 'United States',
    'ארה"ב': 'United States',
    'ארה"ב, ישראל': 'Israel',
    'פורטוגל': 'Portugal',
    'טורקיה': 'Turkey',
    'הונגריה, אנגליה': 'Hungary',
    'הונגריה, ארה"ב': 'Hungary',
    'צרפת': 'France'
}

# Update the dictionary values to English locations
for key in data:
    hebrew_location = data[key][1]
    data[key] = (data[key][0], country_translation.get(hebrew_location, hebrew_location))


# Function to apply data based on author_name
def fill_missing(row):
    if pd.isna(row['birth_year']) and row['author_name'] in data:
        row['birth_year'] = data[row['author_name']][0]
    if pd.isna(row['country']) and row['author_name'] in data:
        row['country'] = data[row['author_name']][1]
    return row

# Apply the function row-wise
df = df.apply(fill_missing, axis=1)


In [6]:
def interpolate_across_books(group):
    birth = group['birth_year'].iloc[0]
    death = group['death_year'].iloc[0]

    if pd.isna(birth):
        group['interpolated_year'] = np.nan
        return group

    # Assume 60-year lifespan if death_year is missing
    if pd.isna(death):
        death = birth + 60

    # Set start and end years
    start_year = birth + 30
    end_year = death


    # Get max unit_index for each book
    max_by_book = group.groupby('book_name')['unit_index'].transform('max')

    # Compute relative position
    group['relative_position'] = group['unit_index'] / max_by_book

    # Interpolate year based on relative position
    group['interpolated_year'] = start_year + group['relative_position'] * (end_year - start_year)

    return group

# Apply to each author
df = df.groupby('author_name', group_keys=False).apply(interpolate_across_books)

# Create a new 'year' column
df['year'] = df['interpolated_year']
# Create a column indicating how old someone is when they die.
df['age_at_death']=df['death_year']-df['birth_year']

/tmp/ipykernel_455/3700194629.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('author_name', group_keys=False).apply(interpolate_across_books)


In [7]:
df['age_at_death'].describe()
# Count how many observations have age_at_death <= 30
count_under_30 = (df['age_at_death'] <= 30).sum()

print(f"Number of observations with age_at_death <= 30: {count_under_30}")

authors_under_30 = df.loc[df['age_at_death'] <= 30, 'author_name'].unique()
print(authors_under_30)

mask = df['age_at_death'] <= 30
cols = ['author_name', 'interpolated_year', 'age_at_death', 'birth_year', 'death_year']

out = (
    df.loc[mask, cols]
      .assign(interpolated_year=lambda d: d['interpolated_year'].round(0).astype('Int64'))  # tidy display
      .sort_values(['author_name', 'interpolated_year', 'death_year'])
)

print(out.to_string(index=False))


Number of observations with age_at_death <= 30: 248
["ר' שמחה בונים סופר"]
       author_name  interpolated_year  age_at_death  birth_year  death_year
ר' שמחה בונים סופר               1871          29.0      1842.0      1871.0
ר' שמחה בונים סופר               1871          29.0      1842.0      1871.0
ר' שמחה בונים סופר               1871          29.0      1842.0      1871.0
ר' שמחה בונים סופר               1871          29.0      1842.0      1871.0
ר' שמחה בונים סופר               1871          29.0      1842.0      1871.0
ר' שמחה בונים סופר               1871          29.0      1842.0      1871.0
ר' שמחה בונים סופר               1871          29.0      1842.0      1871.0
ר' שמחה בונים סופר               1871          29.0      1842.0      1871.0
ר' שמחה בונים סופר               1871          29.0      1842.0      1871.0
ר' שמחה בונים סופר               1871          29.0      1842.0      1871.0
ר' שמחה בונים סופר               1871          29.0      1842.0      1871.0
ר' שמחה בונים

In [8]:
#Changing word length to numeric and summarizing it
df['word_length'] = pd.to_numeric(df['word_length'], errors='coerce')
df['word_length'].info()

<class 'pandas.core.series.Series'>
Index: 156915 entries, 0 to 156914
Series name: word_length
Non-Null Count   Dtype  
--------------   -----  
155049 non-null  float64
dtypes: float64(1)
memory usage: 6.4 MB


In [9]:
print(df.info())
#Filtering out to get rid of some obvious citations
df_filtered = df[~df['book_name'].str.contains('הערות|הגהות', na=False)] # Changed this line
print(df_filtered.info())

#df_filtered = df_filtered[(df_filtered['word_length'] >= 11)]
#print(df_filtered['word_length'].info())





<class 'pandas.core.frame.DataFrame'>
Index: 156915 entries, 0 to 156914
Data columns (total 22 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Unnamed: 0              152909 non-null  float64
 1   author_name             156915 non-null  object 
 2   book_name               156914 non-null  object 
 3   unit                    156776 non-null  object 
 4   word_length             155049 non-null  float64
 5   birth_year              156915 non-null  float64
 6   death_year              128434 non-null  float64
 7   city                    128327 non-null  object 
 8   country                 156915 non-null  object 
 9   other_cities            4343 non-null    object 
 10  approx_birth            24545 non-null   float64
 11  approx_death            8367 non-null    float64
 12  approx_location         3793 non-null    float64
 13  mapped_id               132847 non-null  float64
 14  author_name_stats_file  1

In [10]:
#Examining very short observations to verify that they are not in fact responsa

# Filter rows with word_count between 8 and 10 inclusive
subset8_10 = df_filtered[(df_filtered['word_length'] >= 8) & (df_filtered['word_length'] <= 10)]
subset11_15 = df_filtered[(df_filtered['word_length'] >= 11) & (df_filtered['word_length'] <= 15)]

# Count how many observations
count8_10 = len(subset8_10)
count11_15 = len(subset11_15)
print(f"Number of observations with word_count between 8 and 10: {count8_10}")
print(f"Number of observations with word_count between 11 and 15: {count11_15}")

#

Number of observations with word_count between 8 and 10: 85
Number of observations with word_count between 11 and 15: 176


In [11]:
import pandas as pd
import numpy as np
from tabulate import tabulate
from docx import Document
from docx.shared import Inches

# -------------------------------
# Define region mapping (updated)
# -------------------------------

region_map = {
    "Eastern Europe": [
        "Russia", "Poland", "Ukraine", "Lithuania", "Latvia", "Estonia",
        "Belarus", "Hungary", "Czech Republic", "Slovakia", "Romania", "Bulgaria"
    ],
    "Central and Western Europe": [
        "France", "Germany", "Netherlands", "Belgium", "Switzerland",
        "Austria", "United Kingdom", "England", "Ireland"
    ],
    "Italy and the Balkans": [
        "Italy", "Greece", "Yugoslavia", "Bosnia",
        "Serbia", "Croatia", "Bosnia-Herzegovina"
    ],
     "Iberia": [
        "Spain", "Portugal",
    ],
    "North Africa": [
        "Morocco", "Algeria", "Tunisia", "Libya", "Egypt"
    ],
    "Israel/Palestine": [
        "Israel", "Palestine", "Ottoman Palestine", "Mandatory Palestine",
        "Eretz Israel", "ישראל"
    ],
    "West Asia (ex Israel)": [
        # Arabia
        "Saudi Arabia", "Yemen", "Oman", "United Arab Emirates", "Arabia", "Hejaz",
        # Syria/Iraq/Iran/Turkey
        "Syria", "Turkey", "Iraq", "Iran", "Mesopotamia", "Persia",
        # Other West Asia
        "Lebanon", "Jordan", "Qatar", "Bahrain", "Kuwait", "Georgia", "Armenia", "Azerbaijan"
    ],
    "North America/Australia": [
        "United States", "USA", "Canada", "Mexico", "Australia"
    ]
}



def map_region(country):
    for region, countries in region_map.items():
        if country in countries:
            return region
    return "Other"

# -----------------------------------
# Assume df_filtered is already loaded
# -----------------------------------
# For example:
# df_filtered = pd.read_stata("your_file.dta")

# -----------------------------------
# Compute statistics
# -----------------------------------

num_responsa = len(df_filtered)
unique_authors = df_filtered['author_name'].nunique()
avg_obs_per_author = num_responsa / unique_authors
median_obs_per_author = df_filtered['author_name'].value_counts().median()
avg_word_length = df_filtered['word_length'].mean()

# Time-based shares
df_time = df_filtered[['birth_year', 'source']].dropna() # Include 'source' column in df_time
geonim_responsa = df_time[df_time['source'] == 'dataset_2'].shape[0]
rishonim_responsa = df_time[(df_time['source'] == 'dataset_1') & (df_time['birth_year'] <= 1480)].shape[0]
acharonim_responsa = df_time[df_time['birth_year'] > 1480].shape[0]
share_geonim = (geonim_responsa / num_responsa) * 100
share_rishonim = (rishonim_responsa / num_responsa) * 100
share_acharonim = (acharonim_responsa / num_responsa) * 100

# Author birth stats
df_births = df_filtered[['author_name', 'birth_year']].dropna().drop_duplicates()
earliest_birth = int(df_births['birth_year'].min())
average_birth = round(df_births['birth_year'].mean(), 0)
median_birth = round(df_births['birth_year'].median(), 0)
latest_birth = int(df_births['birth_year'].max())
# Region mapping
df_filtered['region'] = df_filtered['country'].apply(map_region)
region_counts = df_filtered['region'].value_counts()
region_shares = (region_counts / num_responsa * 100)

# -----------------------------------
# Assemble summary table
# -----------------------------------

rows = [
    ("Number of responsa", f"{num_responsa:,}"),
    ("Number of unique authors", f"{unique_authors:,}"),
    ("Earliest author birth year", f"{earliest_birth}"),
    ("Median author birth year", f"{median_birth:.0f}"),
    ("Latest author birth year", f"{latest_birth:.0f}"),
    ("Avg. responsa per author", f"{avg_obs_per_author:.0f}"),
    ("Median responsa per author", f"{median_obs_per_author:.0f}"),
    ("Avg. word length", f"{avg_word_length:.0f}"),
    ("Share of Geonic responsa", f"{share_geonim:.1f}%"),
    ("Share of responsa by Rishonim", f"{share_rishonim:.1f}%"),
    ("Share of responsa by Acharonim", f"{share_acharonim:.1f}%")
]

for region, share in region_shares.items():
    rows.append((f"Share of responsa from {region}", f"{share:.1f}%"))

summary_df = pd.DataFrame(rows, columns=["Statistic", "Result"])

# -----------------------------------
# Print a professional-looking table
# -----------------------------------

title = "Table 1. Responsa Project Summary Statistics by Time Period and Geographic Region"
underline = "-" * len(title)

print(f"\n{title}\n{underline}\n")
for i, row in enumerate(summary_df.itertuples(index=False), start=1):
    stat, result = row
    print(f"{i:>2}. {stat:<50} {result:>15}")

# -----------------------------------
# Export to Word document
# -----------------------------------

doc = Document()
doc.add_heading("Responsa Dataset Report", level=1)

# Add table title
doc.add_paragraph(title, style='Heading 2')

# Create and fill the table
table = doc.add_table(rows=1, cols=2)
table.style = 'Table Grid'
hdr_cells = table.rows[0].cells
hdr_cells[0].text = 'Statistic'
hdr_cells[1].text = 'Result'

for row in rows:
    stat, result = row
    row_cells = table.add_row().cells
    row_cells[0].text = str(stat)
    row_cells[1].text = str(result)
    # Clear any existing text, then add a bold run
    row_cells[0].text = ""
    row_cells[0].paragraphs[0].add_run(str(stat)).bold = True

row_cells[1].text = ""

# Optional note
doc.add_paragraph("Note: Responsa are counted by observation, not by unique author. Observations with 10 words or less are excluded.", style='Intense Quote')

# Save the document
doc.save("/content/gdrive/MyDrive/Responsa/responsa_summary_report.docx")
print("\n✅ Word report saved as 'responsa_summary_report.docx'")


/tmp/ipykernel_455/165750954.py:87: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['region'] = df_filtered['country'].apply(map_region)



Table 1. Responsa Project Summary Statistics by Time Period and Geographic Region
---------------------------------------------------------------------------------

 1. Number of responsa                                         134,343
 2. Number of unique authors                                       321
 3. Earliest author birth year                                     680
 4. Median author birth year                                      1784
 5. Latest author birth year                                      1960
 6. Avg. responsa per author                                       419
 7. Median responsa per author                                     172
 8. Avg. word length                                              1274
 9. Share of Geonic responsa                                      2.5%
10. Share of responsa by Rishonim                                14.7%
11. Share of responsa by Acharonim                               82.8%
12. Share of responsa from Eastern Europe            

In [12]:
from docx import Document
from docx.shared import Pt, Inches
from docx.oxml.ns import qn
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT

# Create Word document
doc = Document()

# Set document title
doc.add_heading("Responsa Dataset Report", level=1)

# Add table title and caption
title = "Table 1. Responsa Project by Time Period and Geographic Region"
doc.add_paragraph(title, style='Heading 2')

# Create table with column headers
table = doc.add_table(rows=1, cols=2)
table.style = 'Table Grid'
hdr_cells = table.rows[0].cells
hdr_cells[0].text = 'Statistic'
hdr_cells[1].text = 'Result'

# Set column widths and font styles
for row in summary_df.itertuples(index=False):
    stat, result = row
    row_cells = table.add_row().cells
    row_cells[0].text = str(stat)
    row_cells[1].text = str(result)

# Set column widths (optional fine-tuning)
for row in table.rows:
    row.cells[0].width = Inches(4)
    row.cells[1].width = Inches(2)

# Add caption note (optional)
doc.add_paragraph("Note: Responsa are analyzed by observation, not by unique author.", style='Intense Quote')

# Save the document
doc.save("aresponsa_summary_report.docx")
print("✅ Word report saved as 'responsa_summary_report.docx'")


✅ Word report saved as 'responsa_summary_report.docx'


In [13]:
# Flatten all mapped countries into one list
mapped_countries = set(sum(region_map.values(), []))

# Get unique countries in your data
all_countries = set(df_filtered['country'].dropna().unique())

# Identify countries not in any defined region
other_countries = sorted(all_countries - mapped_countries)

print("Countries in 'Other' category:")
for country in other_countries:
    print("-", country)


Countries in 'Other' category:


In [14]:
# Filter for authors born before 1100 with non-missing birth_year and author_name
early_authors_df = df_filtered[df_filtered['birth_year'] < 1100][['author_name', 'country', 'birth_year']].dropna(subset=['author_name'])

# Drop duplicates to get one row per author
early_authors_df = early_authors_df.drop_duplicates(subset='author_name').sort_values(by='birth_year')

# Display the count
print(f"Number of authors born before 1100: {early_authors_df.shape[0]}\n")

# Print each author's name and country
for row in early_authors_df.itertuples(index=False):
    name = row.author_name
    country = row.country if pd.notna(row.country) else "Unknown"
    print(f"- {name} ({country})")


Number of authors born before 1100: 13

- ר' אחאי משבחא (Iraq)
- הלכות קצובות (Iraq)
- ר' יהודאי גאון (Iraq)
- תשובות הגאונים (Iraq)
- ר' עמרם גאון (Iraq)
- החילוקים (Iraq)
- ר' שמעון קיירא (Iraq)
- ר' האי גאון (Iraq)
- ר' גרשום מאור הגולה (Germany)
- ר' יצחק אלפסי (Morocco)
- ר' שלמה ב"ר יצחק לבית לוי (France)
- ר' שלמה יצחקי (France)
- ר' יוסף הלוי אבן מיגאש (Spain)


In [15]:
# Four-part regional classification (Ashkenaz, Italian, Sepharad, West Asia)

four_part_regions = {
    "Ashkenaz": [
        # Eastern Europe
        "Russia", "Poland", "Ukraine", "Lithuania", "Latvia", "Estonia",
        "Belarus", "Hungary", "Czech Republic", "Slovakia", "Romania", "Bulgaria",

        # Central and Western Europe
        "France", "Germany", "Netherlands", "Belgium", "Switzerland",
        "Austria", "United Kingdom", "England", "Ireland",

        # North America / Australia
        "United States", "USA", "Canada", "Mexico", "Australia"
    ],

    "Italian": [
        "Italy", "Greece", "Yugoslavia", "Bosnia",
        "Serbia", "Croatia", "Bosnia-Herzegovina"
    ],

    "Sepharad": [
        # Iberia
        "Spain", "Portugal",

        # North Africa
        "Morocco", "Algeria", "Tunisia", "Libya", "Egypt",

        # Israel / Palestine
        "Israel", "Palestine", "Ottoman Palestine", "Mandatory Palestine",
        "Eretz Israel", "ישראל",

        # Moved here
        "Turkey", "Syria"
    ],

    "West Asia": [
        # Arabian Peninsula
        "Saudi Arabia", "Yemen", "Oman", "United Arab Emirates", "Arabia", "Hejaz",

        # Core regions (excluding Turkey, Syria)
        "Iraq", "Iran", "Mesopotamia", "Persia",

        # Additional
        "Lebanon", "Jordan", "Qatar", "Bahrain", "Kuwait",
        "Georgia", "Armenia", "Azerbaijan"
    ]
}

# Reverse mapping: country -> region
country_to_region = {
    country: region
    for region, countries in four_part_regions.items()
    for country in countries
}

# Function to classify a country
def classify_country(country):
    return country_to_region.get(country, "Unknown")

# Example usage
test_countries = ["Turkey", "Syria", "Iran", "Poland"]
for c in test_countries:
    print(c, "->", classify_country(c))

Turkey -> Sepharad
Syria -> Sepharad
Iran -> West Asia
Poland -> Ashkenaz


In [16]:
df_filtered.to_csv('/content/gdrive/MyDrive/Responsa/bi_plus_geonim.csv', index=False)


In [17]:

# Save to file (optional)
# -------------------------------
df_filtered.to_excel("bi_plus_geonim_with_year.xlsx", index=False)
